# 00 — Data Ingest
Downloads all pothole datasets, converts to unified YOLO layout, and writes metadata CSV.
Do not re-run this notebook if data/processed/potholes_yolo/ is already populated.

In [ ]:
import os
import shutil
import zipfile
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

RAW_KAGGLE     = Path("data/raw/external/kaggle")
RAW_ROBOFLOW   = Path("data/raw/external/roboflow")
PROCESSED_BASE = Path("data/processed/potholes_yolo")
SPLITS         = ["train", "val", "test"]

for split in SPLITS:
    (PROCESSED_BASE / "images" / split).mkdir(parents=True, exist_ok=True)
    (PROCESSED_BASE / "labels" / split).mkdir(parents=True, exist_ok=True)

print("Directories ready.")

## 1. Download — Kaggle
Requires KAGGLE_USERNAME and KAGGLE_KEY in .env

In [ ]:
import kaggle

kaggle.api.authenticate()
kaggle.api.dataset_download_files(
    "chitholian/annotated-potholes-dataset",
    path=str(RAW_KAGGLE),
    unzip=True
)
print(f"Kaggle files downloaded to {RAW_KAGGLE}")
print(list(RAW_KAGGLE.rglob("*"))[:20])

## 2. Download — Roboflow
Requires ROBOFLOW_API_KEY in .env

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=os.getenv("ROBOFLOW_API_KEY"))
project = rf.workspace("pothole-detection").project("pothole-detection-q5zoj")
dataset = project.version(1).download(
    "yolov8",
    location=str(RAW_ROBOFLOW)
)
print(f"Roboflow files downloaded to {RAW_ROBOFLOW}")

## 3. Merge into unified YOLO layout
Prefixes each filename with its source dataset name to allow per-source filtering during evaluation.

In [ ]:
def merge_yolo_source(
    src_images: Path,
    src_labels: Path,
    split: str,
    prefix: str
) -> list[dict]:
    records = []
    img_exts = {".jpg", ".jpeg", ".png"}
    images_out = PROCESSED_BASE / "images" / split
    labels_out = PROCESSED_BASE / "labels" / split

    for img_path in src_images.iterdir():
        if img_path.suffix.lower() not in img_exts:
            continue
        new_name = f"{prefix}_{img_path.name}"
        shutil.copy2(img_path, images_out / new_name)

        label_path = src_labels / (img_path.stem + ".txt")
        new_label_name = f"{prefix}_{img_path.stem}.txt"
        if label_path.exists():
            shutil.copy2(label_path, labels_out / new_label_name)
        else:
            (labels_out / new_label_name).write_text("")

        records.append({
            "filename": new_name,
            "source": prefix,
            "split": split,
        })
    return records

## 4. Run merge for each source and split
Adjust the paths below if your downloaded folder structure differs.
Check RAW_KAGGLE and RAW_ROBOFLOW after download and update these paths to match actual subdirectory names.

In [ ]:
all_records = []

# --- Kaggle ---
# Update these paths after inspecting RAW_KAGGLE contents
kaggle_splits = {
    "train": (RAW_KAGGLE / "train" / "images", RAW_KAGGLE / "train" / "labels"),
    "val":   (RAW_KAGGLE / "valid" / "images", RAW_KAGGLE / "valid" / "labels"),
}
for split, (imgs, lbls) in kaggle_splits.items():
    if imgs.exists():
        records = merge_yolo_source(imgs, lbls, split, prefix="kaggle")
        all_records.extend(records)
        print(f"Kaggle {split}: {len(records)} images merged")
    else:
        print(f"WARNING: {imgs} not found — check RAW_KAGGLE structure")

# --- Roboflow ---
roboflow_splits = {
    "train": (RAW_ROBOFLOW / "train" / "images", RAW_ROBOFLOW / "train" / "labels"),
    "val":   (RAW_ROBOFLOW / "valid" / "images", RAW_ROBOFLOW / "valid" / "labels"),
    "test":  (RAW_ROBOFLOW / "test"  / "images", RAW_ROBOFLOW / "test"  / "labels"),
}
for split, (imgs, lbls) in roboflow_splits.items():
    if imgs.exists():
        records = merge_yolo_source(imgs, lbls, split, prefix="roboflow")
        all_records.extend(records)
        print(f"Roboflow {split}: {len(records)} images merged")
    else:
        print(f"WARNING: {imgs} not found — check RAW_ROBOFLOW structure")

print(f"\nTotal merged: {len(all_records)} images")

## 5. Write metadata CSV

In [ ]:
meta_df = pd.DataFrame(all_records)
meta_path = Path("data/interim/metadata/datasets_meta.csv")
meta_path.parent.mkdir(parents=True, exist_ok=True)
meta_df.to_csv(meta_path, index=False)
print(meta_df.groupby(["source", "split"]).size().reset_index(name="count"))

## 6. Verify final layout

In [ ]:
for split in SPLITS:
    n_images = len(list((PROCESSED_BASE / "images" / split).glob("*")))
    n_labels = len(list((PROCESSED_BASE / "labels" / split).glob("*.txt")))
    print(f"{split:6s} — images: {n_images:4d}  labels: {n_labels:4d}")